# WP47 — Recursive Self-Description

**Prometheus v0 PoC · Work Package 47**

> *"What is a self, if not a being that perceives itself?"*
> — Douglas Hofstadter, *I Am a Strange Loop*, 2007

## What WP47 does

Hofstadter's central claim: a **self** arises when a system has a model of itself that is *rich enough to refer back to the model*.

WP47 gives Prometheus exactly this:

1. **DependencyGraph** — built by parsing the system's own source code (`prometheus/wp*.py`).  No hardcoded data.
2. **SelfDescription** — generated purely from graph introspection: module count, dependency edges, foundation layers, top-layer integrators, longest chain.
3. **SelfVerifier** — runs 6 invariants on the description analogous to WP27's `InvariantGuard`, but applied to the self-model itself.
4. **Hofstadter property** — the system's model of itself is *part* of the system.  The description refers to the system that generated it.

The self-description that passes all invariants is the verified representation of Prometheus.


In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import logging
logging.basicConfig(level=logging.WARNING)

from prometheus.wp47_self_description import (
    build_self_description,
    verify_wp47_exit_criteria,
    DependencyGraph,
)
print("WP47 loaded ✓")


## Build the Self-Description

The system reads its own source files and constructs a verified self-model.

In [ ]:
report = build_self_description()
print(report.summary())


## The Verified Self-Description Text

This text was generated by reading `prometheus/wp*.py` — no hardcoded strings.

In [ ]:
print(report.verified_text)


## Dependency Graph Visualisation

Which WPs import from which?

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use("Agg")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

graph = report.graph
nodes = graph.nodes

if HAS_MPL:
    # Simple tier plot: x = topological layer, y = WP number within layer
    topo = report.description.topo_layers
    pos = {}
    for layer_i, layer in enumerate(topo):
        for node_j, wp in enumerate(sorted(layer)):
            pos[wp] = (layer_i, node_j - len(layer) / 2.0)

    fig, ax = plt.subplots(figsize=(14, 8))

    # Draw edges
    for wp, node in nodes.items():
        if wp not in pos:
            continue
        for dep in node.imports:
            if dep in pos:
                x0, y0 = pos[wp]
                x1, y1 = pos[dep]
                ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                            arrowprops=dict(arrowstyle="->", color="gray", alpha=0.4, lw=0.8))

    # Draw nodes
    leaf_set  = set(report.description.leaf_wps)
    integ_set = set(report.description.integrator_wps)
    for wp, (x, y) in pos.items():
        if wp in integ_set:
            color = "gold"
        elif wp in leaf_set:
            color = "lightgreen"
        else:
            color = "lightblue"
        circ = plt.Circle((x, y), 0.35, color=color, ec="gray", zorder=3)
        ax.add_patch(circ)
        ax.text(x, y, f"WP{wp}", ha="center", va="center", fontsize=6, fontweight="bold", zorder=4)

    ax.set_xlim(-0.8, len(topo) - 0.2)
    ax.set_ylim(-max(len(l) for l in topo) / 2.0 - 1, max(len(l) for l in topo) / 2.0 + 1)
    ax.set_xlabel("Topological Layer (0 = foundation)")
    ax.set_title(f"Prometheus Dependency Graph  |  {len(nodes)} modules  |  {report.description.n_dependency_edges} edges")

    from matplotlib.patches import Patch
    legend = [Patch(facecolor="lightgreen", label="Foundation (leaf)"),
              Patch(facecolor="gold",       label="Top integrator"),
              Patch(facecolor="lightblue",  label="Intermediate")]
    ax.legend(handles=legend, fontsize=9, loc="upper right")
    ax.axis("off")

    plt.tight_layout()
    plt.savefig("wp47_dependency_graph.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Plot saved → wp47_dependency_graph.png")
else:
    print("matplotlib not available — text summary only")
    for wp in sorted(nodes):
        imports = nodes[wp].imports
        print(f"  WP{wp:02d} → {['WP' + str(i) for i in imports] if imports else ['(leaf)']}")


## Key Graph Statistics

In [ ]:
d = report.description
print(f"Modules found          : {d.n_modules}")
print(f"Dependency edges       : {d.n_dependency_edges}")
print(f"Foundation modules     : WP{sorted(d.leaf_wps)}")
print(f"Top integrators        : WP{d.integrator_wps}")
print(f"Longest chain depth    : {d.longest_chain_len}")
print(f"Chain path             : {' → '.join(f'WP{wp}' for wp in d.longest_chain_path)}")
print(f"Topological layers     : {len(d.topo_layers)}")
print()
print("Topological layers:")
for i, layer in enumerate(d.topo_layers):
    print(f"  Layer {i}: WP{sorted(layer)}")


## Hofstadter Statement

In [ ]:
print(report.hofstadter_statement)


## WP47 Exit Criteria

In [ ]:
criteria = verify_wp47_exit_criteria(report)
all_pass = all(criteria.values())
print(f"{'PASS' if all_pass else 'FAIL'} — WP47 Exit Criteria")
print()
for name, result in criteria.items():
    status = "✓" if result else "✗"
    print(f"  [{status}] {name}")
print()
print(f"All criteria pass: {all_pass}")


## Conclusion

WP47 instantiates Hofstadter's **"I Am a Strange Loop"**:

- The system's **model of itself** (DependencyGraph) is built from its own source code.
- That model is **part of the system** (in the `prometheus` package).
- The description refers back to the system that generated it — the loop closes.

**Cross-reference with WP42**: WP47 shows the system *can* produce a verified self-description.  WP42 shows that self-description has **irreducible blind spots** (the Gödel sentence).  Both are true simultaneously — consistent with Gödel's First Incompleteness Theorem.
